In [1]:
from pathlib import Path
import pandas as pd
import gzip
import json

### Exploring MAOMAO sequence cards

This notebook shows how to load the released sequence-card collection, inspect one complete peptide card, and perform a simple endpoint query.

- Locate the cards

In [2]:
candidate_paths = [
    Path("../../sequence_profiles/sequence_cards.jsonl.gz"),
    Path("../sequence_profiles/sequence_cards.jsonl.gz"),
    Path("sequence_profiles/sequence_cards.jsonl.gz"),
]

CARDS_PATH = next(
    (path.resolve() for path in candidate_paths if path.is_file()),
    None,
)

if CARDS_PATH is None:
    raise FileNotFoundError(
        "Could not locate sequence_profiles/sequence_cards.jsonl.gz"
    )

print("Sequence cards:", CARDS_PATH)

Sequence cards: /home/nicole/Descargas/maomao/sequence_profiles/sequence_cards.jsonl.gz


- Inspect one sequence card: By default, the first card in the collection is displayed. A specific card can instead be retrieved by setting `CARD_ID`.

In [3]:
CARD_ID = None  # Example: "sha256_..."

card = None

with gzip.open(CARDS_PATH, "rt", encoding="utf-8") as handle:
    for line in handle:
        current_card = json.loads(line)

        if CARD_ID is None or current_card["id"] == CARD_ID:
            card = current_card
            break

if card is None:
    raise KeyError(f"No sequence card was found for ID: {CARD_ID}")

print(json.dumps(card, indent=2, ensure_ascii=False))

{
  "schema_version": "1.0.0",
  "resource_version": "1.1.0",
  "id": "sha256_c70d828a62440a19acb4c85c0d1cff29a5b340ac5ba671e7d5869d2e754783db",
  "sequence": "AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS",
  "length": 33,
  "activity_summary": {
    "ambiguous": [
      "toxic",
      "hemolytic"
    ],
    "negative": [
      "cytotoxic",
      "anti_mammalian_cells"
    ],
    "no_information": [
      "cytolysis",
      "neurotoxic",
      "embryotoxic",
      "ichthyotoxic"
    ]
  },
  "evidence_counts": {
    "toxic": {
      "positive": 3,
      "negative": 1
    },
    "cytotoxic": {
      "negative": 1
    },
    "hemolytic": {
      "positive": 4,
      "negative": 8,
      "unlabeled": 1
    },
    "anti_mammalian_cells": {
      "negative": 2
    }
  },
  "ontology": {},
  "negative_evidence": {
    "contributes_to_ambiguity": {
      "toxic": {
        "sources": [
          {
            "source": "tAMPer",
            "categories": [
              "weak or unconfirmed negatives"
 

- Activity summary: The table combines each final endpoint status with its direct-source counts and any ontology support recorded in the card.

In [4]:
activity_rows = []

for status, endpoints in card["activity_summary"].items():
    for endpoint in endpoints:
        counts = card["evidence_counts"].get(endpoint, {})
        ontology = card["ontology"].get(endpoint)

        if ontology:
            ontology_effect = (
                f"{ontology['direct_status']} → {ontology['final_status']} "
                f"(support: {', '.join(ontology['support_from'])})"
            )
        else:
            ontology_effect = None

        activity_rows.append({
            "endpoint": endpoint,
            "final_status": status,
            "positive_sources": counts.get("positive", 0),
            "negative_sources": counts.get("negative", 0),
            "unlabeled_sources": counts.get("unlabeled", 0),
            "conflicting_sources": counts.get("conflicting", 0),
            "ontology": ontology_effect,
        })

activity_summary = pd.DataFrame(activity_rows).sort_values("endpoint")
display(activity_summary)

,endpoint,final_status,positive_sources,negative_sources,unlabeled_sources,conflicting_sources,ontology
3,anti_mammalian_cells,negative,0,2,0,0,None
4,cytolysis,no_information,0,0,0,0,None
2,cytotoxic,negative,0,1,0,0,None
6,embryotoxic,no_information,0,0,0,0,None
1,hemolytic,ambiguous,4,8,1,0,None
7,ichthyotoxic,no_information,0,0,0,0,None
5,neurotoxic,no_information,0,0,0,0,None
0,toxic,ambiguous,3,1,0,0,None


- Additional card information: `toxicity_targets` describes affected organisms or biological targets supported by direct positive evidence. It does not represent the organism of peptide origin.

In [5]:
target_rows = [
    {
        "endpoint": endpoint,
        "reported_target": target,
    }
    for endpoint, targets in card["toxicity_targets"].items()
    for target in targets
]

toxicity_targets = pd.DataFrame(
    target_rows,
    columns=["endpoint", "reported_target"],
)

toxicity_properties = pd.DataFrame(card["toxicity_properties"])

physicochemical_summary = pd.DataFrame(
    card["physicochemical_summary"].items(),
    columns=["descriptor", "value"],
)

print("Affected organisms / toxicity targets")
display(toxicity_targets)

print("Quantitative toxicity properties")
display(toxicity_properties)

print("Physicochemical summary")
display(physicochemical_summary)

Affected organisms / toxicity targets


,endpoint,reported_target


Quantitative toxicity properties


""


Physicochemical summary


,descriptor,value
0,net_charge_pH,2.998495
1,boman_index,-0.020303
2,fcr,0.181818
3,aa_entropy,2.990623


- Simple card query: The following example retrieves up to ten cards with a selected final endpoint status.

In [6]:
ENDPOINT = "neurotoxic"
STATUS = "positive"
MAX_RESULTS = 10

matches = []

with gzip.open(CARDS_PATH, "rt", encoding="utf-8") as handle:
    for line in handle:
        current_card = json.loads(line)

        endpoints_with_status = current_card[
            "activity_summary"
        ].get(STATUS, [])

        if ENDPOINT in endpoints_with_status:
            matches.append({
                "id": current_card["id"],
                "sequence": current_card["sequence"],
                "length": current_card["length"],
                "final_status": STATUS,
                "quantitative_measurements": len(
                    current_card["toxicity_properties"]
                ),
            })

        if len(matches) == MAX_RESULTS:
            break

matches = pd.DataFrame(matches)
display(matches)

,id,sequence,length,final_status,quantitative_measurements
0,sha256_5958364b758eff160ee866452fd341c43eaf4fa...,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,35,positive,0
1,sha256_2c475890e78b2fc16c34a3ad1c59cd9ab2faa21...,AAAISCVGSPECPPKCRAQGCKNGKCMNRKCECYYC,36,positive,0
2,sha256_ed4da599cb41fed5e529543c21e2dde474e515a...,AACKCDDEGPDIRTAPLTGTVDLGSCNAGWEKCASYYTIIADCCRKKK,48,positive,0
3,sha256_98dc6a87e4166aa098cb5a1f100297e7fdf5526...,AACLGMFESCDPNNDKCCPNRECNRKHKWCKYKLW,35,positive,0
4,sha256_4fcb4182aabded96c474982c49c6f28be2320f9...,AACYSSDCRVKCRAMGFSSGKCIDSKCKCYK,31,positive,0
5,sha256_99bc769deaf2ac9091c0cc910f838928f941623...,AACYSSDCRVKCVAMGFSSGKCINSKCKCYK,31,positive,0
6,sha256_877d478be9bd890d2d9bb00bcc2e9f1b0183fb3...,AAKYCKLPVRYGPCKKKIPSFYYKWKAKQCLPFDYSGCGGNANRFK...,57,positive,0
7,sha256_71f68386219f39cae9ab5390709d1af57ad0d05...,AAPCFCPGKPDRGDLWILRGTCPGGYGYTSNCYKWPNICCYPH,43,positive,0
8,sha256_50a1534c0ec57dd750f4ce2dc4b63da9c590724...,AAPCFCSGKPGRGDLWILRGTCPGGYGYTSNCYKWPNICCYPH,43,positive,0
9,sha256_a27c299c82f1b37534f4a7524e62843486668ba...,AAPCSCPGKPGRGDLWIFRGTCPGGYGYTSNCYKWPNICCYPH,43,positive,0
